# Finding Bottlenecks with PyTorch Profiler
---

In the previous notebook we measured our training loop and found it slower than expected.  In this notebook we will reach for our first profiling tool — `torch.profiler` — to find out *which section* of the training step is eating the time.  We will identify the problem, apply a fix, and verify the improvement.

## Baseline: How Slow Is It?

Let's re-run [`train_v1.py`](../source_code/intro/train_v1.py) to get a clean baseline measurement.  This is a straightforward CIFAR-10 + ResNet18 FP32 training loop on a single GPU — nothing exotic.

In [ ]:
!python ../source_code/intro/train_v1.py

**Expected output:**

```
steps timed: 55  mean step: ~260 ms  throughput: ~985 img/s
```

~985 images per second on a modern GPU.  That feels slow.  An NVIDIA L4 can push tens of thousands of images per second through ResNet18 at full utilisation.  Something is wrong — but the timing alone does not tell us what.

This is exactly when we reach for the profiler.

## Introducing `torch.profiler`

The PyTorch Profiler is a context manager that wraps your training loop and records the time spent in each operation.  The key API is:

```python
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    schedule=torch.profiler.schedule(wait=1, warmup=5, active=10, repeat=1),
    on_trace_ready=torch.profiler.tensorboard_trace_handler('/workspace/logs/my_run'),
    record_shapes=True,
) as prof:
    for step, (x, y) in enumerate(loader):
        # ... training step ...
        prof.step()   # ← tell the profiler a step just finished
```

**Key parameters:**

- `activities` — record both CPU-side Python calls and CUDA kernel launches
- `schedule` — skip the first `wait` steps, spend `warmup` steps calibrating, then actively record `active` steps; avoids capturing noisy early steps
- `on_trace_ready` — write a TensorBoard-compatible trace file when the active window closes
- `prof.step()` — **must** be called at the end of every step so the profiler knows where step boundaries are

The profiler adds some overhead (~5–10 %), so we only record a subset of steps rather than the whole run.

## Running with the Profiler

[`train_v1_profile.py`](../source_code/intro/train_v1_profile.py) is identical to [`train_v1.py`](../source_code/intro/train_v1.py) with the profiler wrapper added.  Let's run it and collect a trace.

In [ ]:
!python ../source_code/intro/train_v1_profile.py

**Expected output:**

```
steps timed: 55  mean step: ~278 ms  throughput: ~921 img/s
Trace written to /workspace/logs/train_v1_profile
```

The step time is slightly higher than the plain run — that is the profiler's own overhead.  The trace file has been written to `/workspace/logs/train_v1_profile/`.

## Viewing the Trace in TensorBoard

TensorBoard is already running on port 8889.  Open it in your browser:

**http://localhost:8889**

Select the **PyTorch Profiler** plugin from the top navigation, then choose `train_v1_profile` from the run selector on the left.

The **Overview** tab shows a step timeline broken down by section.  You should see something like this:

```
┌─────────────────────────────────────────────────────────────────┐
│  Step breakdown (mean across recorded steps)                    │
│                                                                 │
│  DataLoader  ████████████████████████████████████  ~240 ms      │
│  Forward             ██  ~12 ms                                 │
│  Backward         █████  ~18 ms                                 │
│  Optimizer             █  ~3 ms                                 │
└─────────────────────────────────────────────────────────────────┘
```

The **DataLoader section consumes roughly 90 % of every step.**  The GPU is sitting idle while the CPU loads and preprocesses each batch on a single thread before the next forward pass can begin.

## The Problem: Single-Threaded Data Loading

Look at these two settings in [`train_v1.py`](../source_code/intro/train_v1.py):

```python
NUM_WORKERS = 0      # ← Bug: data loading happens on the main thread
PIN_MEMORY  = False  # ← Bug: batches land in pageable memory
```

With `num_workers=0` every batch is decoded, augmented, and assembled by the same Python thread that runs the training loop.  The GPU has to wait for the CPU to finish before it can start the next forward pass.

With `pin_memory=False` the DataLoader allocates tensors in ordinary pageable memory.  Copying pageable memory to the GPU requires an intermediate staging buffer in pinned memory and cannot be overlapped with GPU compute.

**The fix is two lines:**

```python
NUM_WORKERS = 4      # spawn 4 worker processes to load data in parallel
PIN_MEMORY  = True   # allocate directly in pinned memory for fast H→D copies
```

Pytorch DataLoader workers run in separate processes, so they can decode and augment the *next* batch while the GPU is running the *current* forward pass — the CPU and GPU work in parallel instead of alternating.

## Apply the Fix

[`train_v1_fixed.py`](../source_code/intro/train_v1_fixed.py) is [`train_v1_profile.py`](../source_code/intro/train_v1_profile.py) with those two lines changed.  Let's run it and collect a new trace.

In [ ]:
!python ../source_code/intro/train_v1_fixed.py

**Expected output:**

```
steps timed: 55  mean step: ~94 ms  throughput: ~2727 img/s
Trace written to /workspace/logs/train_v1_fixed
```

Refresh TensorBoard and select the `train_v1_fixed` run.  The step timeline should now look very different — DataLoader has shrunk to a small slice and GPU compute fills most of each step.

## Comparing Before and After

| Script | Mean step | Throughput | DataLoader share |
|---|---|---|---|
| [`train_v1.py`](../source_code/intro/train_v1.py) (buggy) | ~260 ms | ~985 img/s | ~90 % |
| [`train_v1_fixed.py`](../source_code/intro/train_v1_fixed.py) | ~94 ms | ~2727 img/s | small |

**~2.8× speedup from two lines of code.**

The profiler told us *which section* was slow.  The fix was straightforward once we knew where to look.

But this was a simple case — the profiler handed us the answer.  In the next notebook we will encounter a problem where the profiler shows us that something is wrong, but cannot tell us *what* or *why*.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red; height:80px;"><b><br/>[Next Notebook — AMP and the Limits of the Profiler](intro-amp.ipynb)</b></div></center>

---

## Links and Resources

- [PyTorch Profiler documentation](https://pytorch.org/docs/stable/profiler.html)
- [PyTorch Profiler with TensorBoard tutorial](https://pytorch.org/tutorials/intermediate/tensorboard_profiler_tutorial.html)
- [PyTorch Profiler recipe](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0).